# Cortex Continuous Learning RL\n\nNotebook Colab pour exécuter le package `cortex/` avec Sentinel RL, agents auto-apprenants et pipeline continu.

## 1. Setup

In [ ]:
!pip -q install torch pandas networkx matplotlib httpx\n

In [ ]:
import sys\nfrom pathlib import Path\n\nROOT = Path('/content/coco')\nif ROOT.exists() and str(ROOT) not in sys.path:\n    sys.path.insert(0, str(ROOT))\n\nfrom cortex.training_pipeline import run_training\nfrom cortex.training_pipeline import build_verified_colab_payload, save_verified_colab_payload, push_verified_colab_payload\nfrom cortex.evaluation import compute_metrics, plot_training\n

## 2. Run Training Pipeline

In [ ]:
results = run_training(episodes=10, num_events=120, export_dir='/content/cortex_exports')\nruntime_df = results['runtime_df']\nreward_curve = results['reward_curve']\nruntime_df.head()\n

## 3. Evaluation

In [ ]:
metrics = compute_metrics(runtime_df, reward_curve)\nmetrics\n

## 4. Visualization

In [ ]:
plot_training(runtime_df, reward_curve, '/content/cortex_exports')\n

## 5. Verified Cortex Sync

In [ ]:
RUN_ID = 'colab-run-continuous-001'\nTRAINING_PLAN_ID = 'plan-continuous-learning-001'\nTARGET_AGENTS = ['decision', 'sentinel', 'threat_hunter', 'trust']\nKNOWLEDGE_REGISTRY_FINGERPRINT = 'known-registry-update-me'\nREVIEWER = 'analyst@cortex.local'\nNOTES = 'Verified continuous learning run. Advisory risk signals only.'\n\nverified_payload = build_verified_colab_payload(\n    runtime_df,\n    metrics,\n    run_id=RUN_ID,\n    training_plan_id=TRAINING_PLAN_ID,\n    target_agents=TARGET_AGENTS,\n    knowledge_registry_fingerprint=KNOWLEDGE_REGISTRY_FINGERPRINT,\n    reviewer=REVIEWER,\n    notes=NOTES,\n)\nverified_payload_path = save_verified_colab_payload(verified_payload, '/content/cortex_exports')\nverified_payload_path, verified_payload['dataset_fingerprint'], len(verified_payload['accepted_item_ids'])\n

In [ ]:
CORTEX_SYNC_URL = 'http://127.0.0.1:18082/v1/training/colab/ingest'\nCORTEX_SHARED_SECRET = ''\nENABLE_PUSH = False\n\npush_result = {'status': 'disabled'}\nif ENABLE_PUSH:\n    if not CORTEX_SHARED_SECRET:\n        raise ValueError('CORTEX_SHARED_SECRET must be set before ENABLE_PUSH=True')\n    push_result = push_verified_colab_payload(\n        verified_payload,\n        url=CORTEX_SYNC_URL,\n        secret=CORTEX_SHARED_SECRET,\n    )\npush_result\n

## 6. Exported Artifacts

In [ ]:
!ls -lah /content/cortex_exports\n